# Unsupervised Machine Learning and Time Series Forecasting of Alphabet Inc. (GOOGL)

**Module:** Machine Learning 2 — Continuous Assessment (individual)

**Chosen domain:** Stock market — the only permitted source, Yahoo Finance, restricted to the last 10 years

**Dataset:** `data/GOOGL.csv` — daily OHLCV prices for Alphabet Inc. (GOOGL), **1 September 2016 to 29 September 2026**, 2,532 trading days, downloaded from Yahoo Finance with the `yfinance` package

## Introduction, motivation, problem domain, assessment objectives and the rationale for the chosen dataset

### Introduction

This assessment applies **unsupervised machine learning** and **time series forecasting** to ten years of
daily share prices for Alphabet Inc. (GOOGL), the parent company of Google. There are two parts:

1. **Clustering.** A share does not behave the same way every day. Some days are quiet, others are wild.
   Nobody gives us labels for these "market regimes", so finding them is an **unsupervised** problem. We
   group the trading days using two clustering algorithms - **K-Means** and **K-Medoids** - and compare
   what they find.
2. **Time series forecasting.** We then treat the closing price as a time series, look for trend,
   seasonality and anomalies, build an **ARIMA** model, extend it to a seasonal **SARIMA** model, and check
   whether either one can forecast better than a very simple guess.

### Motivation

* **Risk management.** A 2% fall means something different on a calm day than on a panic day. If we can put
  each day into a regime, we can change how much risk we take.
* **Forecasting.** Investors want to know the future price *and* how uncertain it is. ARIMA gives both a
  prediction and a confidence interval. If the model cannot beat a simple guess, that is also a useful
  answer — it supports the **efficient market hypothesis** (Fama, 1970).

### Description of the problem domain

The domain is the **stock market**. Every trading day the exchange gives us six numbers:

| Variable | Meaning |
|---|---|
| `Open` | Price of the first trade of the day |
| `High` | Highest price during the day |
| `Low` | Lowest price during the day |
| `Close` | Price of the last trade of the day |
| `Adj Close` | Close adjusted for splits and dividends — the correct one for returns |
| `Volume` | Number of shares traded |

Two facts about share prices shape everything we do:

* **Prices are non-stationary, but returns are roughly stationary.** The price keeps trending upwards, which
  breaks the assumptions of most models. The *percentage change* behaves much better. This is why ARIMA has
  a differencing step, and why our clustering features are all percentages instead of raw prices.
* **Volatility comes in bunches.** Big moves follow big moves. That is exactly what makes "market regime" a
  sensible idea, and it is why clustering is worth trying.

### Assessment objectives

| # | Objective |
|---|---|
| **O1** | Clean the data and get at least 7 columns of mixed type (categorical, continuous, discrete) |
| **O2** | Choose clustering algorithms, explain why, implement two of them and compare |
| **O3** | Measure **similarity/dissimilarity between clusters** |
| **O4** | Explain the difference between **silhouette score** and **Davies–Bouldin index**, and say which solution we prefer |
| **O5** | Explore the time series for **trend, seasonality and anomalies** |
| **O6** | Find the ARIMA parameters **(p, d, q)**, evaluate the forecast, give strengths and limitations |

### Questions we ask about this dataset

* **Q1.** Which clustering solution do we prefer, and why?
* **Q2.** Can ARIMA or SARIMA beat a simple guess?

### Why this dataset was chosen

1. **It follows the rules.** It comes from Yahoo Finance and covers exactly ten years, as the brief requires.
2. **It is big enough.** 2,532 trading days is plenty of rows for clustering and for ARIMA.
3. **The regimes really exist.** These ten years include the 2018 tech selloff, the COVID crash of 2020, the
   2021 boom, the 2022 fall (GOOGL lost 39.1% that year) and the 2023–2026 AI rally. Clustering only makes
   sense if the data really contains different behaviours, and this period does.
4. **The data is clean.** Alphabet is a huge company that trades every day, so there are no missing days and
   no strange values to argue about.
5. **We can build mixed variable types.** From the six raw columns we can make continuous, discrete and
   categorical variables, as the brief requires.
6. **It supports both parts of the assignment.** The same daily prices feed the clustering analysis (regimes from returns, volatility and volume) and the time series analysis (trend, ARIMA and SARIMA forecasting), so one dataset answers both sets of questions.

In [3]:
import os
import yfinance as yf

import pandas as pd

In [4]:
# Getting 10 years daily Dataset of Apple from Yahoo Finanace

os.makedirs("data", exist_ok=True)

# auto_adjust=False keeps the "Adj Close" column, as in your file

df = yf.download("GOOGL", start="2016-09-01", end="2026-09-30", auto_adjust=False)
df.to_csv("data/GOOGL.csv")
df.head()

[*********************100%***********************]  1 of 1 completed


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,GOOGL,GOOGL,GOOGL,GOOGL,GOOGL,GOOGL
Date,,,,,,
2016-09-01,39.197018,39.570000,39.644501,39.316502,39.598999,26070000
2016-09-02,39.467941,39.843498,39.855000,39.662998,39.763500,26998000
2016-09-06,40.020184,40.401001,40.544498,39.771500,39.919498,39790000
2016-09-07,40.018692,40.399502,40.529999,40.186001,40.396500,22906000
2016-09-08,39.763615,40.141998,40.421001,40.050499,40.261002,23554000


#### Data Loading

In [5]:
cols = ['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
df_raw = pd.read_csv('data/GOOGL.csv', skiprows=3, names=cols)

display(df_raw.shape)

display(df_raw.head())

(2532, 7)

,Date,Adj Close,Close,High,Low,Open,Volume
0,2016-09-01,39.197018,39.570000,39.644501,39.316502,39.598999,26070000
1,2016-09-02,39.467941,39.843498,39.855000,39.662998,39.763500,26998000
2,2016-09-06,40.020184,40.401001,40.544498,39.771500,39.919498,39790000
3,2016-09-07,40.018692,40.399502,40.529999,40.186001,40.396500,22906000
4,2016-09-08,39.763615,40.141998,40.421001,40.050499,40.261002,23554000


In [6]:
display(df_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 2532 entries, 0 to 2531
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       2532 non-null   str    
 1   Adj Close  2532 non-null   float64
 2   Close      2532 non-null   float64
 3   High       2532 non-null   float64
 4   Low        2532 non-null   float64
 5   Open       2532 non-null   float64
 6   Volume     2532 non-null   int64  
dtypes: float64(5), int64(1), str(1)
memory usage: 138.6 KB


None

#### Data Cleaning

In [7]:
df = df_raw.copy()
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
df = df.set_index('Date').sort_index()

num_cols = ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

In [8]:
display(df.info())

<class 'pandas.DataFrame'>
DatetimeIndex: 2532 entries, 2016-09-01 to 2026-09-29
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Close  2532 non-null   float64
 1   Close      2532 non-null   float64
 2   High       2532 non-null   float64
 3   Low        2532 non-null   float64
 4   Open       2532 non-null   float64
 5   Volume     2532 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 138.5 KB


None

In [9]:
display(df.shape)

df = df[~df.index.duplicated(keep='first')].dropna()

display(df.shape)

display(df.describe())

(2532, 6)

(2532, 6)

,Adj Close,Close,High,Low,Open,Volume
count,2532.000000,2532.000000,2532.000000,2532.000000,2532.000000,2.532000e+03
mean,121.673956,122.479583,123.805541,121.050751,122.398992,3.374918e+07
std,82.054660,82.090928,83.068542,80.984017,82.011212,1.509569e+07
min,37.306007,37.660999,38.525002,37.179501,38.095001,9.312000e+06
25%,57.914528,58.465625,58.987375,57.902750,58.446502,2.423958e+07
50%,102.620003,103.596500,104.778500,101.930000,103.063751,3.006330e+07
75%,147.624104,148.944874,151.447495,147.825005,149.558750,3.811588e+07
max,402.121185,402.619995,408.609985,396.359985,399.920013,1.331780e+08
